# 03 · Análisis de negocio — ModaModerna


In [45]:
# Cargar librerías

# Librerías nativas de Python
import os

# Manipulación y análisis de datos
import numpy as np
import pandas as pd

# Gráficos y visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar opciones para que Pandas no corte las columnas ni la pantalla
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Tema global de gráficos
sns.set_theme(style='whitegrid', palette='colorblind')


In [46]:
# Cargar las tablas limpias
# Los ids se leen como texto para mantener su formato
# y las fechas como datetime.

# Cargar las tablas limpias

clientes_limpio = pd.read_csv('datos_limpios/clientes_limpio.csv')
productos_limpio = pd.read_csv('datos_limpios/productos_limpio.csv')
campana_limpio = pd.read_csv('datos_limpios/campana_limpio.csv')
pedidos_limpio = pd.read_csv('datos_limpios/pedidos_limpio.csv')
lineas_pedido_limpio = pd.read_csv('datos_limpios/lineas_pedido_limpio.csv')
devoluciones_limpio = pd.read_csv('datos_limpios/devoluciones_limpio.csv')
visitas_web_limpio = pd.read_csv('datos_limpios/visitas_web_limpio.csv')

# Construcción de la tabla maestra mediante merge

Para construir la tabla maestra, primero identificamos la **tabla de hechos con mayor nivel de detalle**.

En este caso, es `lineas_pedido`, ya que cada fila representa una línea concreta de un pedido, es decir, un producto concreto dentro de un pedido.

A partir de ella establecemos las conexiones con las tablas que necesitamos:

- `lineas_pedido` se conecta con `productos` mediante `producto_id`.
- `lineas_pedido` se conecta con `pedidos` mediante `pedido_id`.
- `lineas_pedido` se conecta con `devoluciones` mediante `linea_id`.
- `pedidos` se conecta con `clientes` mediante `cliente_id`.

Si no podemos conectar directamente todas las tablas con la tabla de hechos, buscamos cómo se relacionan las otras tablas y utilizamos esas relaciones para completar la tabla maestra.

La tabla `campana` y `visitas_web` no son necesarias para las preguntas de negocio planteadas, por lo que no se incorporan a la tabla maestra.

Usamos `how='left'` para mantener todas las líneas de pedido, aunque no exista información relacionada en alguna de las tablas.


In [47]:
# Paso 1: añadimos a cada línea la información del producto.
# La relación se realiza mediante producto_id.

maestro = lineas_pedido_limpio.merge(
    productos_limpio,
    on='producto_id',
    how='left'
)


In [48]:
# Paso 2: añadimos a cada línea la información del pedido.
# La relación se realiza mediante pedido_id.

maestro = maestro.merge(
    pedidos_limpio,
    on='pedido_id',
    how='left'
)


In [49]:
# Paso 3: añadimos a cada línea la información del cliente.
# La relación se realiza mediante cliente_id,
# que hemos obtenido al unir previamente con pedidos.

maestro = maestro.merge(
    clientes_limpio,
    on='cliente_id',
    how='left'
)


In [50]:
# Paso 4: añadimos la información de las devoluciones.
# La relación se realiza mediante linea_id.

maestro = maestro.merge(
    devoluciones_limpio[['linea_id', 'fecha_devolucion', 'motivo']],
    on='linea_id',
    how='left'
)


# El resultado es un DataFrame maestro con toda la información que necesitamos para el análisis.
print(maestro.shape)
maestro.head()

(181637, 25)


,linea_id,pedido_id,producto_id,talla,cantidad,precio_unitario,descuento_pct,nombre,categoria,temporada,coleccion,coste_unitario,precio_base,cliente_id,fecha_pedido,campana_id,canal,ciudad_x,metodo_pago,fecha_registro,ciudad_y,genero,edad,fecha_devolucion,motivo
0,900001,500001,1567,XS,1,20.86,0.5,Vestido Terra,Vestidos,Otoño/Invierno 2025,Playa Club,8.68,21.42,106480,14/07/2025,2.0,instagram,Vic,tarjeta,2022-07-11,Badalona,M,33.0,NaN,NaN
1,900002,500002,1463,S,1,66.70,0.0,Bolso Ona,Accesorios,Otoño/Invierno 2025,Urban SS25,31.30,69.21,123135,2025-11-13,NaN,google,Barcelona,paypal,2024-08-27,Tarragona,F,32.0,NaN,NaN
2,900003,500002,1313,M,1,35.42,0.0,Cinturón Gala,Complementos,Otoño/Invierno 2025,Urban SS25,16.50,33.94,123135,2025-11-13,NaN,google,Barcelona,paypal,2024-08-27,Tarragona,F,32.0,NaN,NaN
3,900004,500002,1565,XS,1,44.57,0.0,Camiseta Iris,Camisetas,Otoño/Invierno 2025,Punto & Lana,23.13,46.60,123135,2025-11-13,NaN,google,Barcelona,paypal,2024-08-27,Tarragona,F,32.0,NaN,NaN
4,900005,500003,1393,S,1,53.11,0.0,Camiseta Gala,Camisetas,Primavera/Verano 2025,Denim,25.07,55.17,127034,2025-09-04,NaN,directo,Sant Cugat del Vallès,bizum,2022-09-12,Barcelona,M,24.0,NaN,NaN


## Creación de nuevas columnas en Maestro

In [51]:
# Antes de analizar, creamos tres columnas nuevas en el maestro porque
# necesitamos información calculada que no tenemos directamente en los datos.

# Ingreso generado por cada línea después del descuento.
# descuento_pct es una fracción entre 0 y 1.

maestro['ingreso_linea'] = (
    maestro['cantidad']
    * maestro['precio_unitario']
    * (1 - maestro['descuento_pct'])
)

# Indica si la línea tiene una devolución asociada.
# Si tiene devolución → True = 1.
# Si no tiene devolución → False = 0.

maestro['devuelta'] = maestro['motivo'].notna()

# Ingreso correspondiente a las líneas devueltas.
# True = 1 → mantiene el ingreso.
# False = 0 → devuelve 0.
# Las líneas devueltas mantienen su ingreso y las demás tienen 0.

maestro['ingreso_devuelto'] = (maestro['ingreso_linea'] * maestro['devuelta'])

# Para sumar los ingresos totales, los ingresos de las líneas devueltas y los ingresos después de devoluciones, podemos usar el método `sum()` de Pandas.

print("Ingreso total:", maestro['ingreso_linea'].sum())
print("Ingreso de líneas devueltas:", maestro['ingreso_devuelto'].sum())
print("Ingreso después de devoluciones:", maestro['ingreso_linea'].sum() - maestro['ingreso_devuelto'].sum())


Ingreso total: 11673072.857
Ingreso de líneas devueltas: 2909026.1969999997
Ingreso después de devoluciones: 8764046.66


# 1. ¿Qué se vende?


In [52]:
# Tenemos **productos en el catálogo**, pero no todos necesariamente han registrado ventas.
# Comprobamos cuántos productos tienen al menos una línea de pedido.


# Número total de productos disponibles en el catálogo.

total_productos = productos_limpio['producto_id'].nunique()

# Número de productos que aparecen en las líneas de pedido.

productos_con_ventas = maestro['producto_id'].nunique()

# Productos del catálogo que no han registrado ventas.

productos_sin_ventas = total_productos - productos_con_ventas

print(f"Productos del catálogo: {total_productos}")
print(f"Productos con ventas: {productos_con_ventas}")
print(f"Productos sin ventas: {productos_sin_ventas}")

# Todos nuestros productos registran ventas, por lo que no hay productos sin ventas.


Productos del catálogo: 600
Productos con ventas: 600
Productos sin ventas: 0


## Productos con ventas



In [53]:
# Además del número total, podemos ver qué productos han registrado ventas y cuánto han generado.

# Calculamos los ingresos generados por cada producto.
# groupby → agrupar por producto
# ['ingreso_linea'] → elegir qué columna queremos sumar
# sum() → sumar
# sort_values() → ordenar

ventas_por_producto = (
    maestro
    .groupby(['producto_id', 'nombre'], as_index=False)['ingreso_linea']
    .sum()
    .sort_values('ingreso_linea', ascending=False)
)

ventas_por_producto.head(10)


,producto_id,nombre,ingreso_linea
526,1527,Vestido Cel,40669.446
87,1088,Vestido Ona,39395.688
231,1232,Zapato Iris,39376.263
466,1467,Zapato Nora,38430.853
325,1326,Cinturón Kira,38314.974
3,1004,Bolso Ona,37731.302
239,1240,Camiseta Alba,37433.914
71,1072,Camiseta Gala,37307.418
274,1275,Camiseta Gala,37211.644
38,1039,Zapato Flor,37106.781


# 2. ¿Cuáles son los productos TOP?


In [54]:
# Consideramos el TOP productos al **20 % de los 600 productos del catálogo**:
# **600 × 20 % = 120 productos TOP**
# El 20 % es el criterio que escogemos como referencia, siguiendo el principio de Pareto, que plantea que una parte pequeña de los elementos (20% de los productos) puede concentrar una parte importante de los resultados ( 80% de ingresos).

n_productos_top = round(total_productos * 0.20)

print(f"Productos TOP: {n_productos_top}")

productos_top = ventas_por_producto.head(n_productos_top)

productos_top.head(10)

Productos TOP: 120


,producto_id,nombre,ingreso_linea
526,1527,Vestido Cel,40669.446
87,1088,Vestido Ona,39395.688
231,1232,Zapato Iris,39376.263
466,1467,Zapato Nora,38430.853
325,1326,Cinturón Kira,38314.974
3,1004,Bolso Ona,37731.302
239,1240,Camiseta Alba,37433.914
71,1072,Camiseta Gala,37307.418
274,1275,Camiseta Gala,37211.644
38,1039,Zapato Flor,37106.781


In [65]:
# Calculamos qué porcentaje de los ingresos generan los productos TOP.

ingresos_totales = ventas_por_producto['ingreso_linea'].sum()

ingresos_top = productos_top['ingreso_linea'].sum()

porcentaje_top = ingresos_top / ingresos_totales * 100


print(f"Los ingresos totales son {ingresos_totales}, los ingresos de los productos TOP son {ingresos_top:.1f} y representan el {porcentaje_top:.1f}% de los ingresos totales.")

Los ingresos totales son 11673072.857, los ingresos de los productos TOP son 3883977.2 y representan el 33.3% de los ingresos totales.


# 3. ¿Se cumple Pareto?

El principio de Pareto plantea como referencia una distribución aproximada de: 20 % de los productos → 80 % de los ingresos.

Los 120 productos TOP (20 % del catálogo) generan el 33,3 % de los ingresos.

Por tanto, no se cumple Pareto y los ingresos están más repartidos entre los productos.

Esto significa que el negocio no depende de unos pocos productos para generar la mayor parte de sus ingresos.

# 4. ¿Qué productos TOP concentran más devoluciones? ¿Por qué se devuelven?



In [67]:
# isin() → nos quedamos solo con los 120 productos TOP.
# groupby() → agrupamos por producto.
# .sum() → contamos cuántas devoluciones tiene cada producto (True = 1).
# sort_values() → ordenamos de más a menos devoluciones.
# head(10) → mostramos los 10 primeros solo para visualizar, no porque el análisis se limite a 10.

devoluciones_por_producto = (
    devoluciones_top[devoluciones_top['devuelta']]
    .groupby(['producto_id', 'nombre', 'motivo'])['devuelta']
    .sum()
    .sort_values(ascending=False)
)

devoluciones_por_producto.head(10)

producto_id  nombre          motivo          
1058         Bolso Gala      talla incorrecta    31
1509         Cinturón Nácar  talla incorrecta    30
1583         Bolso Gala      talla incorrecta    29
1042         Abrigo Flor     talla incorrecta    29
1266         Camiseta Lluna  talla incorrecta    28
1151         Jersey Nit      talla incorrecta    28
1104         Vestido Nit     talla incorrecta    28
1432         Pantalón Lluna  talla incorrecta    28
1357         Pantalón Mila   talla incorrecta    27
1220         Vestido Aura    talla incorrecta    27
Name: devuelta, dtype: int64

# CONCLUSIONES

- Los ingresos totales son **11.673.072,86 €**. Los productos TOP generan **3.883.977,20 €**, que representan el **33,3 % de los ingresos totales**.

- El negocio **no depende de unos pocos productos para generar la mayor parte de sus ingresos**.

- Entre los principales motivos de devolución encontramos la **talla incorrecta**, que concentra una parte importante de las devoluciones de los productos TOP.

- Esto puede indicar que **conviene revisar el tallaje o la información sobre las tallas** para reducir este tipo de devoluciones.